# What Diluvium adds to Lua

Six additions, all of them in the 5.5 line. Every one is a **syntax
error in stock Lua**, and none of them takes a reserved word — so
existing code using `switch` or `defer` as a variable name keeps
working.

Source compatibility with stock Lua is absolute and is promised not to
change. These are additions, not a dialect you have to opt into.

## 1. String interpolation

`$"..."` interpolates. A slot holds any expression.

In [ ]:
local name, n = "ada", 3
print($"hello {name}, {n} times")
print($"{n} squared is {n * n}, and n > 2 is {tostring(n > 2)}")

Everything after `::` is handed to `string.format`. It is `::` rather
than the `:` other languages use because `:` already introduces a method
call — and `$"{obj:method()}"` has to keep meaning that.

In [ ]:
local pi, count = math.pi, 42
print($"pi is {pi::%.4f}")
print($"padded: {count::%05d}   hex: {count::%x}")

local s = "abc"
print($"a method call still works: {s:upper()}")

## 2. Safe navigation

`a?.b` and `a?[k]`. Once the value left of a `?` is nil the **whole
remaining chain is skipped** — nothing further is indexed or called —
and the result is nil.

In [ ]:
local config = { server = { host = "localhost", port = 8080 } }

print(config?.server?.host)
print(config?.database?.host)      -- no error: the chain stopped
print(config?["server"]?["port"])

It tests **nil, not falsiness**. `false?.x` still raises, because
`false` is a value that genuinely cannot be indexed.

In [ ]:
local f = false
print(pcall(function() return f?.x end))

## 3. Null coalescing and compound assignment

`??=` assigns only when the target is nil. Note `false` is left alone —
the same nil-not-falsiness rule.

In [ ]:
local a = nil
a ??= "a default"

local b = false
b ??= "not used"

print(a, b)

The full set: `+= -= *= /= //= %= ^= |= &= <<= >>= ..= ??=`. There is
no `~=` form, since that already means "not equal".

In [ ]:
local x = 10
x += 5   print(x)
x -= 2   print(x)
x *= 3   print(x)
x //= 4  print(x)

local s = "a"
s ..= "b" ; s ..= "c"
print(s)

**The target's prefix is evaluated once.** `t[next_key()] += 1` calls
`next_key` a single time, which is the whole reason to write it that way
rather than `t[k] = t[k] + 1`.

In [ ]:
local calls = 0
local function key() calls = calls + 1 return "hits" end

local t = { hits = 1 }
t[key()] += 10

print($"value {t.hits}, key() called {calls} time")

## 4. `switch`

The subject is evaluated **once**, a case can list several values, and
there is **no fallthrough**.

In [ ]:
local function describe(n)
  switch n do
    case 0 then return "zero"
    case 1, 2, 3 then return "small"
    case 4, 5, 6 then return "medium"
    default return "large"
  end
end

for _, n in ipairs{ 0, 2, 5, 99 } do print(n, describe(n)) end

Cases match any value, not just numbers.

In [ ]:
local function handle(event)
  switch event.kind do
    case "spawned", "exited" then print("lifecycle: " .. event.kind)
    case "faulted" then print("!! " .. (event.detail or "no detail"))
    default print("ignored: " .. tostring(event.kind))
  end
end

handle{ kind = "spawned" }
handle{ kind = "faulted", detail = "index a nil value" }
handle{ kind = "weather" }

**One consequence worth knowing.** `switch` is a *contextual* keyword,
so it stays usable as a name — but that means the subject cannot start
with `(`, a string, or a table constructor: `switch (x)`, `switch "s"`
and `switch {}` are all function calls in stock Lua and have to keep
meaning that. Bind to a local first.

In [ ]:
local switch = "still a usable name"
print(switch)

## 5. `defer` and `with`

`defer stat` runs a statement **however the block is left** — falling
off the end, `break`, `goto`, `return`, or an error. It desugars to
Lua's to-be-closed variables, so the ordering and unwinding are
inherited rather than invented.

In [ ]:
local function work()
  defer print("  cleanup ran")
  print("  doing the work")
  return "result"
end

print("returning normally:")
print(work())

Including out of an error — which is the case it exists for.

In [ ]:
print(pcall(function()
  defer print("  cleanup still ran")
  error("something went wrong")
end))

Several defers run in **reverse declaration order**, like a stack.

In [ ]:
do
  defer print("third")
  defer print("second")
  defer print("first")
  print("body")
end

`with name = expr do ... end` binds a to-be-closed local directly, for
a value that has a `__close` metamethod. Several bindings may be
separated by commas, and they close in reverse order.

In [ ]:
local function resource(name)
  print("  open " .. name)
  return setmetatable({}, { __close = function() print("  close " .. name) end })
end

with a = resource("outer"), b = resource("inner") do
  print("  body")
end

`defer` survives a `coroutine.yield` and runs at the right time rather
than at yield time.

In [ ]:
local co = coroutine.create(function()
  defer print("  deferred, after the second resume")
  coroutine.yield("paused")
  print("  resumed")
end)

print(coroutine.resume(co))
print(coroutine.resume(co))

## 6. Everything else is Lua 5.5

Integers and floats are distinct, `//` is integer division, bitwise
operators are built in, and `goto` is there.

In [ ]:
print(math.type(1), math.type(1.0))
print(7 // 2, 7.0 // 2, 7 % 2)
print(5 & 3, 5 | 3, 5 ~ 3, 1 << 4)
print(math.maxinteger + 1 == math.mininteger)   -- integers wrap

A bytecode chunk from the 5.4 line will **not** load here: this is a
different Lua release with a different bytecode format. Recompile.
Source is backward compatible, as ever.

Press **Bytecode** on any cell above to see what the compiler made of
it — nothing runs when you do.